# Laboratorio #7

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab7)

Para este laboratorio se trabajó con los datos de **@traficogt** en data.

> **Nota:** Se utilizó python 3.12.6 para compatibilidad con librerias.

## Librerias

In [742]:
# Separación de data y filtración
import json, os, ast
from typing import List, Dict, Tuple, Optional
import pandas as pd
import numpy as np

# Análisis exploratorio
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import seaborn as sns

# Limpieza lenguaje
import re, unicodedata
import spacy

nlp = spacy.load("es_core_news_sm")
STOP_ES = nlp.Defaults.stop_words

# Análisis de tópicos
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA

## Constantes


In [743]:
DATA_DIR = "data/"
DATA_FILE = DATA_DIR + "traficogt.txt"
DATA_CLEAN = DATA_DIR + "clean_traficogt.csv"

DATA_CLEAN_IN  = "data/clean_traficogt.csv"
DATA_CLEAN_OUT = "data/clean_traficogt_v2.csv"

DATA_USE = "data/clean_traficogt_v3.csv"
TOPICS_OUT = "data/clean_traficogt_v4.csv"

DO_S1_2_3 = False
DO_S8 = True

In [744]:
URL_RE   = re.compile(r"https://\S+")
EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF\u2600-\u26FF]+")

## Utils

Función que se reutilizarán en las secciones

In [745]:
def bindZona(text: str) -> str:
    """
    Une menciones de zona en un token estable: 'zona_10'.
    Soporta 'zona 10', 'z10', 'Z 10', etc.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\b(?:zona|z)\s*([0-9]{1,2})\b", r"zona_\1", text, flags=re.I)
    return text

In [746]:
def parseLemmas(raw) -> List[str]:
    """
    Convierte la columna 'lemmas' (string tipo lista) a lista real.
    Si no se puede parsear, devuelve [].
    """
    if isinstance(raw, list):
        return raw
    if pd.isna(raw):
        return []
    try:
        v = ast.literal_eval(raw)
        return v if isinstance(v, list) else []
    except Exception:
        return []

In [747]:
def cleanLemmaList(tokens: List[str],
                   extraStop: Optional[set] = None,
                   keepPrefixes: Tuple[str, ...] = ("zona_", "km_")) -> List[str]:
    """
    Limpia lista de lemas:
      - quita números puros
      - quita extras en extraStop
      - conserva tokens con prefijos de dominio (zona_, km_)
    """
    if not tokens:
        return []
    extraStop = extraStop or set()
    out = []
    for t in tokens:
        if not isinstance(t, str) or not t:
            continue
        if t in extraStop:
            continue
        if t.isdigit():
            # descarta números sueltos
            continue
        out.append(t)
    return out

## Sección 1

In [748]:
def processTweetsToCsv(input_path="data/traficogt.txt", output_path="data/clean_traficogt.csv"):
    """
    Lee tweets en formato JSONL (uno por línea), extrae campos específicos
    y escribe un CSV. Si un campo no existe o viene como null, se escribe 'null'
    en el CSV (usando na_rep='null').

    Columnas:
      tweet_id, date, user_id, username, followers_count, friends_count, statuses_count,
      raw_content, reply_count, retweet_count, like_count, quote_count, conversation_id,
      hashtags, mentioned_users, mentioned_users_ids, view_count, place, coordinates,
      in_reply_to_tweet_id, in_reply_to_user_id, in_reply_to_username,
      has_quote, quoted_tweet_id, quoted_user_id, quoted_username
    """

    # Detectar codificación simple (UTF-16 o UTF-8/UTF-8 BOM)
    enc = "utf-8"
    with open(input_path, "rb") as fb:
        sig = fb.read(4)
    if sig.startswith(b"\xff\xfe") or sig.startswith(b"\xfe\xff"):
        enc = "utf-16"
    elif sig.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    cols = [
        "tweet_id", "date", "user_id", "username",
        "followers_count", "friends_count", "statuses_count",
        "raw_content", "reply_count", "retweet_count", "like_count", "quote_count",
        "conversation_id", "hashtags", "mentioned_users", "mentioned_users_ids",
        "view_count", "place", "coordinates",
        "in_reply_to_tweet_id", "in_reply_to_user_id", "in_reply_to_username",
        "has_quote", "quoted_tweet_id", "quoted_user_id", "quoted_username"
    ]

    def extract_hashtags(hts):
        if not isinstance(hts, list):
            return ""
        out = []
        for h in hts:
            if isinstance(h, str):
                out.append(h)
            elif isinstance(h, dict):
                out.append(h.get("name") or h.get("text") or "")
        return ",".join([t for t in out if t])

    def extract_mentioned_users(mus):
        if not isinstance(mus, list):
            return ""
        out = []
        for u in mus:
            if isinstance(u, str):
                out.append(u)
            elif isinstance(u, dict):
                n = u.get("username") or u.get("screen_name") or ""
                if n:
                    out.append(n)
        return ",".join(out)

    def extract_mentioned_user_ids(mus):
        if not isinstance(mus, list):
            return ""
        out = []
        for u in mus:
            if isinstance(u, dict):
                mid = u.get("id")
                if mid is not None:
                    out.append(str(mid))
            elif isinstance(u, (int, str)):
                out.append(str(u))
        return ",".join(out)

    def serialize_obj(obj):
        # Para guardar 'place' o 'coordinates' como JSON en una sola celda
        if obj is None:
            return pd.NA
        if isinstance(obj, (dict, list)):
            return json.dumps(obj, ensure_ascii=False)
        return str(obj)

    rows = []
    with open(input_path, "r", encoding=enc) as f_in:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            try:
                tw = json.loads(line)
            except json.JSONDecodeError:
                continue

            user = tw.get("user") or {}
            in_reply_user = tw.get("inReplyToUser") or {}
            qt = tw.get("quotedTweet")

            # Manejo minimal de quote:
            has_quote = 0
            quoted_tweet_id = pd.NA
            quoted_user_id = pd.NA
            quoted_username = pd.NA
            if qt is not None:
                has_quote = 1
                if isinstance(qt, dict):
                    quoted_tweet_id = qt.get("id", pd.NA)
                    q_user = qt.get("user") or {}
                    quoted_user_id = q_user.get("id", pd.NA)
                    quoted_username = q_user.get("username", pd.NA)
                elif isinstance(qt, (int, str)):
                    quoted_tweet_id = qt  # si viene solo el id

            row = {
                "tweet_id": tw.get("id"),
                "date": tw.get("date"),
                "user_id": user.get("id"),
                "username": user.get("username"),
                "followers_count": user.get("followersCount"),
                "friends_count": user.get("friendsCount"),
                "statuses_count": user.get("statusesCount"),
                "raw_content": tw.get("rawContent"),
                "reply_count": tw.get("replyCount"),
                "retweet_count": tw.get("retweetCount"),
                "like_count": tw.get("likeCount"),
                "quote_count": tw.get("quoteCount"),
                "conversation_id": tw.get("conversationId"),
                "hashtags": extract_hashtags(tw.get("hashtags")),
                "mentioned_users": extract_mentioned_users(tw.get("mentionedUsers")),
                "mentioned_users_ids": extract_mentioned_user_ids(tw.get("mentionedUsers")),
                "view_count": tw.get("viewCount"),
                "place": serialize_obj(tw.get("place")),
                "coordinates": serialize_obj(tw.get("coordinates")),
                "in_reply_to_tweet_id": tw.get("inReplyToTweetId"),
                "in_reply_to_user_id": in_reply_user.get("id") if isinstance(in_reply_user, dict) else pd.NA,
                "in_reply_to_username": in_reply_user.get("username") if isinstance(in_reply_user, dict) else pd.NA,
                "has_quote": has_quote,
                "quoted_tweet_id": quoted_tweet_id,
                "quoted_user_id": quoted_user_id,
                "quoted_username": quoted_username,
            }

            # Convertir None explícitamente a pd.NA para que se exporte como 'null'
            for k, v in row.items():
                if v is None:
                    row[k] = pd.NA

            rows.append(row)

    df = pd.DataFrame(rows, columns=cols)
    df.to_csv(output_path, index=False, encoding="utf-8", na_rep="null")
    return {"output_csv": output_path, "rows": len(df)}

En el proceso de extracción de datos de los tweets, se seleccionaron 26 columnas del JSON original para preservar en el CSV, priorizando aquellas que contienen información clave para el análisis exploratorio y la construcción de redes.

### Idenficación del tweet

* **tweet_id**: ID único del tweet

  * Sirve para deduplicar, indexar y unir con otras tablas.

* **date**: fecha y hora UTC

  * Sirve para análisis de series temporales (picos por hora/día), relacionar con temporada de lluvias y cambios a lo largo del año.

### Autor (perfil del nodo)

* **user_id, username**: identifican al autor

  * Sirve para construir nodos de la red y etiquetar resultados.

* **followers_count, friends_count, statuses_count**: métricas del autor

  * Sirve para contexto de influencia potencial, comparar engagement relativo (p. ej., likes/followers).

### Contenido (texto y temas)

* **raw_content**: texto original

  * Sirve para tokenización, extracción de toponimia, tópicos y análisis de sentimiento (no mezclar con quoted).

* **hashtags**: lista de temas marcados

  * Sirve para identificar temas frecuentes y co-ocurrencias (wordclouds, top hashtags).

### Interacciones del tweet (engagement)

* **reply_count, retweet_count, like_count, quote_count, view_count**

  * Sirve para medir impacto y alcance, priorizar eventos relevantes y comparar periodos.

### Conversaciones e hilos

* **conversation_id**: agrupa tweets del mismo hilo

  * Sirve para reconstruir discusiones y detectar temas "calientes".

* **in_reply_to_tweet_id, in_reply_to_user_id, in_reply_to_username**: relaciones de respuesta

  * Sirve para construir aristas "reply" en la red (autor -> destinatario).

### Red de menciones

* **mentioned_users, mentioned_users_ids**: usuarios mencionados en el texto

  * Sirve para construir aristas "mention" (autor -> mencionados), detectar cuentas más aludidas y subredes temáticas.

### Ubicación (cuando exista o se infiera)

* **place, coordinates**: geotag oficial (suele ser null)

  * Sirve para mapeo directo; cuando es null, el `raw_content` permite inferir zonas/avenidas y luego unir a polígonos/centroides.

### Citas (quote)

* **has_quote** (0/1), **quoted_tweet_id, quoted_user_id, quoted_username**

  * Sirve para construir aristas "quote" (autor -> citado), identificar fuentes amplificadas y medir centralidad por citas.
  * Nota: el contenido del citado **no** se mezcla con el `raw_content` del autor; se usa solo para la red e influencia.

## Sección 2

In [749]:
def normalizeText(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).lower()
    # quitar saltos de línea
    s = s.replace("\n", " ").replace("\r", " ")
    # quitar urls
    s = URL_RE.sub(" ", s)
    # quitar emojis
    s = EMOJI_RE.sub(" ", s)
    # quitar @ y #
    s = s.replace("@", " ").replace("#", " ")
    # quitar acentos
    s = ''.join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    # normalizar espacios
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [750]:
def applySection2_1() -> pd.DataFrame:
    # cargar csv
    df = pd.read_csv(DATA_CLEAN_IN)

    # aplicar a todas las columnas de texto
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(normalizeText)

    # guardar nuevo csv
    df.to_csv(DATA_CLEAN_OUT, index=False, encoding="utf-8")
    print({"rows_out": len(df), "output": DATA_CLEAN_OUT})
    
    return df

En la limpieza realizada se aplicaron las siguientes transformaciones sobre todas las columnas de texto del archivo:

1. **Conversión a minúsculas**: todo el contenido textual se pasó a minúsculas para uniformar la representación.
2. **Eliminación de saltos de línea**: los caracteres `\n` y `\r` se sustituyeron por espacios, evitando cortes dentro de un mismo campo.
3. **Eliminación de URLs**: se removieron los enlaces que inician con `https://`, evitando ruido en los textos.
4. **Eliminación de emojis**: se quitaron caracteres pertenecientes a los rangos Unicode de símbolos y emojis.
5. **Eliminación de arrobas y numerales**: se suprimieron los símbolos `@` y `#` en cualquier posición del texto.
6. **Eliminación de acentos**: se normalizaron los caracteres con tilde, de modo que `áéíóú` pasaron a `aeiou` y `ñ` a `n`.
7. **Normalización de espacios**: se redujeron múltiples espacios consecutivos a un solo espacio y se recortaron espacios al inicio y al final.

El resultado es un nuevo archivo CSV (`clean_traficogt_v2.csv`) con la misma estructura de columnas, pero con todos los campos de texto **limpios, uniformes y listos para análisis**.

In [751]:
def tokenizeKeepNumbers(text: str):
    doc = nlp(text)
    return [tok.text for tok in doc if tok.is_alpha or tok.like_num]

def lemmatizeKeepNumbers(text: str):
    doc = nlp(text)
    return [tok.lemma_ for tok in doc if (tok.is_alpha or tok.like_num) and tok.lemma_ not in STOP_ES]


In [752]:
def applySection2_2(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # normalizar todas las columnas de texto
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(normalizeText)
    
    # features de conteo
    df['exclaim_count']  = df['raw_content'].str.count(r"[!¡]")
    df['question_count'] = df['raw_content'].str.count(r"[?¿]")
    
    # tokenización y lematización
    df['tokens'] = df['raw_content'].apply(tokenizeKeepNumbers)
    df['lemmas'] = df['raw_content'].apply(lemmatizeKeepNumbers)
    
    # opcional: guardar CSV
    df.to_csv(DATA_USE, index=False, encoding="utf-8")
    print({"rows_out": len(df), "output": DATA_USE})
    
    return df

Después de aplicar esta limpieza y procesamiento de texto, obtendremos un DataFrame enriquecido con nuevas propiedades que permiten un análisis más profundo del contenido textual de los tweets. Específicamente, se generarán:

* **`exclaim_count`**: número de signos de exclamación (`!` o `¡`) en cada tweet, útil para analizar **énfasis o emoción**.
* **`question_count`**: número de signos de interrogación (`?` o `¿`) en cada tweet, útil para detectar **preguntas o curiosidad en el contenido**.
* **`tokens`**: lista de palabras individuales filtradas de caracteres no alfabéticos, que permite análisis de frecuencia, conteo de términos y generación de n-gramas.
* **`lemmas`**: lista de lemas de cada palabra, eliminando stopwords en español, lo que facilita tareas de análisis semántico, agrupación de palabras con la misma raíz y minería de tópicos.

El resultado es un nuevo archivo CSV (`clean_traficogt_v3.csv`)

## Sección 3

### Resumen general

In [753]:
def getDatasetSummary(df: pd.DataFrame) -> dict:
    return {
        "numTweets": len(df),
        "numUsers": df['user_id'].nunique(),
        "numHashtags": df['hashtags'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0).sum(),
        "numMentions": df['mentioned_users'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0).sum()
    }
    
def plotDatasetSummary(df: pd.DataFrame):
    summary = getDatasetSummary(df)
    keys = list(summary.keys())
    values = list(summary.values())
    
    plt.figure(figsize=(8,5))
    sns.barplot(x=keys, y=values, hue=keys, palette="viridis", dodge=False, legend=False)
    plt.title("Resumen General del Dataset")
    plt.ylabel("Cantidad")
    plt.show()


### Conteo de interacciones

In [754]:
def getInteractionsSummary(df: pd.DataFrame) -> dict:
    return {
        "totalReplies": df['reply_count'].sum(),
        "totalRetweets": df['retweet_count'].sum(),
        "totalQuotes": df['quote_count'].sum(),
        "totalLikes": df['like_count'].sum()
    }
    
def plotInteractions(df: pd.DataFrame):
    interactions = getInteractionsSummary(df)
    keys = list(interactions.keys())
    values = list(interactions.values())
    
    plt.figure(figsize=(8,5))
    sns.barplot(x=keys, y=values, hue=keys, palette="magma", dodge=False, legend=False)
    plt.title("Interacciones Totales de Tweets")
    plt.ylabel("Cantidad")
    plt.show()


### Hashtags más frecuentes y menciones

In [755]:
def getTopHashtags(df: pd.DataFrame, topN=10) -> list:
    hashtags = []
    for ht_list in df['hashtags'].dropna():
        hashtags.extend(ht_list.split(','))
    counter = Counter([ht.lower() for ht in hashtags if ht])
    return counter.most_common(topN)

def getTopMentionedUsers(df: pd.DataFrame, topN=10) -> list:
    mentions = []
    for mu_list in df['mentioned_users'].dropna():
        mentions.extend(mu_list.split(','))
    counter = Counter([mu.lower() for mu in mentions if mu])
    return counter.most_common(topN)

In [756]:
def plotTopHashtags(df: pd.DataFrame, topN=10):
    topHashtags = getTopHashtags(df, topN)
    labels, counts = zip(*topHashtags)
    
    plt.figure(figsize=(10,5))
    sns.barplot(x=list(counts), y=list(labels), hue=list(labels), palette="coolwarm", dodge=False, legend=False)
    plt.title(f"Top {topN} Hashtags")
    plt.xlabel("Frecuencia")
    plt.show()

def plotTopMentionedUsers(df: pd.DataFrame, topN=10):
    topUsers = getTopMentionedUsers(df, topN)
    labels, counts = zip(*topUsers)
    
    plt.figure(figsize=(10,5))
    sns.barplot(x=list(counts), y=list(labels), hue=list(labels), palette="Set2", dodge=False, legend=False)
    plt.title(f"Top {topN} Usuarios Mencionados")
    plt.xlabel("Frecuencia")
    plt.show()


### Tweets por hora

In [757]:
def getTweetsByHour(df: pd.DataFrame) -> pd.Series:
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce').dt.tz_convert('America/Guatemala')
    return df.groupby(df['date'].dt.hour).size()

def getTopTweetHours(df: pd.DataFrame, topN=5) -> list:
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce').dt.tz_convert('America/Guatemala')
    counts = df.groupby(df['date'].dt.hour).size()
    return counts.sort_values(ascending=False).head(topN).to_dict()

def plotTweetsByHour(df: pd.DataFrame):
    tweetsByHour = getTweetsByHour(df)
    plt.figure(figsize=(12,5))
    sns.barplot(x=tweetsByHour.index, y=tweetsByHour.values, hue=tweetsByHour.index, palette="viridis", dodge=False, legend=False)
    plt.title("Distribución de Tweets por Hora")
    plt.xlabel("Hora del día")
    plt.ylabel("Número de tweets")
    plt.show()

### Influencia de usuarios (engagement vs seguidores)

In [758]:
def getUserEngagement(df: pd.DataFrame, topN=10) -> pd.DataFrame:
    """
    Calcula engagement por usuario: (likes + retweets + replies) / followers_count.
    Retorna topN usuarios con mayor engagement.
    """
    df_users = df.groupby('user_id').agg({
        'username': 'first',
        'followers_count': 'first',
        'like_count': 'sum',
        'retweet_count': 'sum',
        'reply_count': 'sum'
    }).copy()
    
    df_users['engagement'] = (df_users['like_count'] + df_users['retweet_count'] + df_users['reply_count']) / df_users['followers_count'].replace(0, 1)
    
    return df_users.sort_values('engagement', ascending=False).head(topN)


In [759]:
def plotTopUserEngagement(df: pd.DataFrame, topN=10):
    df_eng = getUserEngagement(df, topN=topN)
    
    if df_eng.empty:
        print("No se encontraron usuarios con engagement.")
        return
    
    plt.figure(figsize=(12,6))
    sns.barplot(
        data=df_eng.reset_index(),
        x='username', 
        y='engagement', 
        hue='username',
        palette="magma",
        dodge=False,
        legend=False
    )
    plt.title(f"Top {topN} Usuarios por Engagement")
    plt.xlabel("Usuario")
    plt.ylabel("Engagement (likes+retweets+replies / followers)")
    plt.xticks(rotation=45, ha='right')
    plt.show()

### Zonas

In [760]:
def extractZones(df: pd.DataFrame, column='raw_content') -> list:
    # Patrón: "Z" o "Zona" seguido de un número (con o sin espacio)
    pattern = r"\b(?:Z|Zona)\s*(\d{1,2})\b"
    zones = []
    for text in df[column].dropna():
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        zones.extend(matches)
    counter = Counter([f"Zona {z}" for z in zones])
    return counter.most_common(10)  # Top 10 zonas

In [761]:
def plotTopZones(df: pd.DataFrame, topN=10):
    topZones = extractZones(df)[:topN]
    if not topZones:
        print("No se encontraron zonas.")
        return
    
    zones, counts = zip(*topZones)
    
    plt.figure(figsize=(10,5))
    sns.barplot(x=list(zones), y=list(counts), hue=list(zones), palette="viridis", dodge=False, legend=False)
    plt.title(f"Top {topN} Zonas Más Mencionadas")
    plt.ylabel("Cantidad de Menciones")
    plt.xlabel("Zona")
    plt.xticks(rotation=45)
    plt.show()

In [762]:
def plotZonesByHour(df: pd.DataFrame, column='raw_content'):
    # Convertir fecha a hora local
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce').dt.tz_convert('America/Guatemala')
    df['hour'] = df['date'].dt.hour

    # Extraer zonas
    pattern = r"\b(?:Z|Zona)\s*(\d{1,2})\b"
    df['zones'] = df[column].dropna().apply(lambda x: re.findall(pattern, x, flags=re.IGNORECASE))
    
    # Crear DataFrame de zona por hora
    rows = []
    for _, row in df.iterrows():
        for z in row['zones']:
            rows.append((int(z), row['hour']))
    if not rows:
        print("No se encontraron zonas con hora.")
        return

    df_zone_hour = pd.DataFrame(rows, columns=['zone', 'hour'])
    
    # Pivot para heatmap
    pivot = df_zone_hour.pivot_table(index='zone', columns='hour', aggfunc='size', fill_value=0)
    
    # Graficar
    plt.figure(figsize=(12,8))
    sns.heatmap(pivot, cmap='viridis', annot=True, fmt="d")
    plt.title("Menciones de Zonas por Hora")
    plt.ylabel("Zona")
    plt.xlabel("Hora del día")
    plt.show()

### Flujo completo

In [763]:
def generateWordCloud(df: pd.DataFrame, column='lemmas', maxWords=100):
    text = ' '.join([' '.join(t) if isinstance(t, list) else str(t) for t in df[column].dropna()])
    wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=maxWords).generate(text)
    plt.figure(figsize=(15, 7))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.show()

In [764]:
def exploratoryTrafficAnalysis(df: pd.DataFrame):
    print("=== Resumen General ===")
    summary = getDatasetSummary(df)
    print(summary)
    
    print("\n=== Interacciones ===")
    interactions = getInteractionsSummary(df)
    print(interactions)
    
    print("\n=== Hashtags Más Frecuentes ===")
    topHashtags = getTopHashtags(df)
    print(topHashtags)
    
    print("\n=== Usuarios Más Mencionados ===")
    topUsers = getTopMentionedUsers(df)
    print(topUsers)
    
    print("\n=== Distribución de Tweets por Hora ===")
    tweetsByHour = getTweetsByHour(df)
    print(tweetsByHour)
    
    print("\n=== Zonas Más Mencionadas ===")
    topZones = extractZones(df)
    print(topZones)

    print("\n=== Engagement por Usuario ===")
    dfEngagement = getUserEngagement(df, topN=10)
    print(dfEngagement)

In [765]:
def exploratoryTrafficAnalysisVisual(df: pd.DataFrame):
    # 1) Resumen
    plotDatasetSummary(df)
    
    # 2) Interacciones
    plotInteractions(df)
    
    # 3) Hashtags top
    plotTopHashtags(df)
    
    # 4) Usuarios mencionados top
    plotTopMentionedUsers(df)
    
    # 5) Tweets por hora
    plotTweetsByHour(df)
    
    # 6) Zonas pop
    plotTopZones(df)
    
    # 7) Top Users por Engagement
    plotTopUserEngagement(df)
    
    # 8) Zonas por hora
    plotZonesByHour(df)
    
    # 9) Nube de palabras
    print("=== Nube de Palabras (Lemas) ===")
    generateWordCloud(df, column='lemmas')


## Sección 4

## Sección 5

## Sección 6

## Sección 7

## Sección 8

### Sentimiento

Aquí queremos obtener una medida simple y trazable del tono de cada tweet (positivo/neutral/negativo) y un indicador de lluvia, para cruzar luego con horas, zonas y tópicos.

**Método.**

* Se usa **VADER** y se amplía el léxico con términos del dominio tráfico/lluvia (p. ej., *accidente, bloqueo, congestion, lluvia, fluido*).
* Para cada tweet se calcula el **`compound`** de VADER y se etiqueta **`sent_label`** con umbrales estándar:

  * `compound >= 0.05` -> **pos**
  * `compound <= -0.05` -> **neg**
  * en otro caso -> **neu**
* **`isRainRelated`** aplica una **heurística por palabras clave** (p. ej., *lluvia, aguacero, inundacion, tormenta*).

**Salidas.**

* Por tweet: `sent_compound` (float), `sent_label` (`pos|neu|neg`), `rain_related` (`True|False`).

In [766]:
def buildSentimentAnalyzer(lexiconEs: Optional[Dict[str, float]] = None) -> SentimentIntensityAnalyzer:
    """
    Construye el analizador VADER y amplía el léxico con términos ES del dominio tráfico.
    """
    try:
        nltk.download('vader_lexicon', quiet=True)
    except Exception:
        pass
    sia = SentimentIntensityAnalyzer()
    baseLex = {
        # negativo tráfico
        'accidente': -2.0, 'choque': -2.0, 'volcadura': -2.5, 'bloqueo': -2.2,
        'manifestacion': -1.5, 'protesta': -1.5, 'trafico': -1.7, 'congestion': -1.7,
        'lento': -0.8, 'parado': -1.2, 'embotellamiento': -2.0, 'tapon': -1.2,
        'lluvia': -1.0, 'aguacero': -1.2, 'inundacion': -2.0,
        'corrupcion': -2.5, 'corrupto': -2.5, 'verguenza': -1.2,
        # positivo tráfico
        'fluido': 1.4, 'libre': 1.2, 'despejado': 1.5, 'mejora': 0.8, 'sintrafico': 2.0
    }
    if lexiconEs:
        baseLex.update(lexiconEs)
    sia.lexicon.update(baseLex)
    return sia


def scoreSentiment(text: str, sia: SentimentIntensityAnalyzer) -> Tuple[float, str]:
    """
    Devuelve (compound, label) con umbrales estándar VADER.
    """
    if not isinstance(text, str) or not text.strip():
        return 0.0, 'neu'
    s = sia.polarity_scores(text)
    c = s['compound']
    lab = 'pos' if c >= 0.05 else 'neg' if c <= -0.05 else 'neu'
    return c, lab


def isRainRelated(text: str) -> bool:
    """
    Heurística simple para lluvia/tormenta.
    """
    if not isinstance(text, str):
        return False
    rainRe = re.compile(r"\b(lluvia|llovizna|aguacero|tormenta|inundacion|chubasco|granizo)\b", re.I)
    return bool(rainRe.search(text))

### Tópicos LDA

Aquí se descubre temas latentes en los tweets para entender de qué se habla (accidentes, bloqueos, política, lluvia, etc.) y luego cruzarlos con **hora/zona/lluvia**.

**Método.**

* Se construye un **corpus** a partir de `lemmas` (o `tokens` si faltan), limpiando números sueltos y ruido (`prepareCorpusForTopics`).
* Se **vectoriza** con `CountVectorizer` usando **unigramas y bigramas** (`ngram_range=(1,2)`), permitiendo tokens con guion bajo (p. ej., `zona_10`) y excluyendo números puros (`vectorizeForTopics`).
* Se entrena **LDA** con `kTopics` tópicos (`fitLdaTopics`) y se asigna a cada tweet su **tópico dominante** por argmax de la distribución documento->tópico (`assignDominantTopic`).
* Se extraen los **términos top** por tópico para su interpretación (`summarizeTopics`).
* Opcionalmente, se genera una **máscara de tráfico** para entrenar LDA sólo en tweets viales/lluvia (`buildTrafficMask`), útil para responder preguntas de @traficogt.

**Entradas.** DataFrame con columnas de texto (`lemmas` o `tokens`).
**Salidas.**

* Por tweet: `topic_id` (entero), y matrices/series auxiliares (p. ej., `topic_share_%`).
* Por tópico: lista de términos representativos (top-N).

**Hiperparámetros clave.**

* `kTopics` (número de temas), `minDf`/`maxDf` (filtrado de términos raros/frecuentes), `ngramRange` (para capturar frases como *calzada_san_juan*, *zona_1*).

**Notas.**

* LDA es **no supervisado**: requiere **interpretación** de los términos top.

In [767]:
def prepareCorpusForTopics(df: pd.DataFrame,
                           lemmaCol: str = 'lemmas',
                           extraStop: Optional[set] = None) -> pd.Series:
    """
    Devuelve una Serie de strings (documentos) lista para vectorizar:
    - usa 'lemmas' si existe; si no, intenta 'tokens'; si no, queda vacío.
    - aplica cleanLemmaList
    """
    if lemmaCol not in df.columns:
        alt = 'tokens' if 'tokens' in df.columns else None
        base = df[alt].fillna('[]').apply(parseLemmas) if alt else pd.Series([[]]*len(df), index=df.index)
    else:
        base = df[lemmaCol].fillna('[]').apply(parseLemmas)

    base = base.apply(lambda xs: cleanLemmaList(xs, extraStop=extraStop))
    docs = base.apply(lambda xs: " ".join(xs))
    return docs


def vectorizeForTopics(docs: pd.Series,
                       minDf: int = 8,
                       maxDf: float = 0.6,
                       ngramRange: Tuple[int, int] = (1, 2)) -> Tuple[CountVectorizer, np.ndarray]:
    """
    Vectoriza con CountVectorizer, excluyendo números puros y permitiendo tokens con '_'.
    """
    # filtra documentos vacíos
    mask = docs.str.strip().astype(bool)
    docsUsed = docs[mask]

    vectorizer = CountVectorizer(
        lowercase=True,
        min_df=minDf,
        max_df=maxDf,
        ngram_range=ngramRange,
        token_pattern=r"(?u)\b(?!\d+$)[\w_]+\b"
    )
    X = vectorizer.fit_transform(docsUsed)
    return vectorizer, (X, mask)


def fitLdaTopics(X, kTopics: int = 6, randomState: int = 42) -> LDA:
    lda = LDA(n_components=kTopics, learning_method='batch', random_state=randomState)
    lda.fit(X)
    return lda


def assignDominantTopic(lda: LDA, X) -> np.ndarray:
    """
    Devuelve el vector topic_id por documento en X (argmax de theta).
    """
    theta = lda.transform(X)
    return theta.argmax(axis=1)


def summarizeTopics(lda: LDA,
                    vectorizer: CountVectorizer,
                    topN: int = 10) -> Dict[int, List[str]]:
    """
    Devuelve dict {topic_id: [top terms]}.
    """
    id2term = np.array(vectorizer.get_feature_names_out())
    topicTerms = {}
    for k, row in enumerate(lda.components_):
        idx = np.argsort(row)[::-1][:topN]
        topicTerms[k] = id2term[idx].tolist()
    return topicTerms


def buildTrafficMask(lemmasTxt: pd.Series) -> pd.Series:
    """
    Devuelve una máscara booleana para dejar tweets de tráfico/lluvia.
    Busca términos de dominio y tokens 'zona_'.
    """
    # términos base (ajusta si deseas)
    trafTerms = [
        r"\btrafico\b", r"\bcongestion\b", r"\baccidente\b", r"\bchoque\b", r"\bbloqueo\b",
        r"\bcalle\b", r"\bavenida\b", r"\bperiferico\b", r"\bcalzada\b", r"\bpuente\b",
        r"\binundacion\b", r"\blluvia\b", r"\baguacero\b", r"\bruta\b", r"\bkm\b",
        r"\bzona_\d{1,2}\b"
    ]
    pat = re.compile("|".join(trafTerms), re.I)
    return lemmasTxt.str.contains(pat)

### Reportes

Al correr el código se generan las métricas de **sentimiento**, **tópicos LDA** y agregados por **hora** / **lluvia**.

**Funciones.**

* `computeAggregates(df)`: calcula resúmenes clave (porcentaje de sentimiento, promedio por hora, efecto de lluvia, share de tópicos).
* `applySection8(...)`: pipeline completo que genera columnas nuevas, entrena LDA, asigna `topic_id`, calcula agregados y **guarda archivos**.

#### Archivos de salida (cuando `saveAux=True`)

1. **Principal**

* `outputCsv` *(por defecto `data/clean_traficogt_v4.csv`)*
  
  Contiene el dataframe enriquecido con columnas nuevas:

  * `hour` (0–23, **America/Guatemala**),
  * `lemmas_list`, `lemmas_txt`,
  * `sent_compound` (float), `sent_label` (`pos|neu|neg`),
  * `rain_related` (`True|False`),
  * `topic_id` (>=0 = tópico asignado; **-1 = sin tópico** si el documento quedó vacío o si se entrenó LDA con `filterTrafficOnly=True` y el tweet no cumple el filtro).

2. **Agregados para el informe** (prefijo = nombre del CSV principal):

* `_sent_overall.csv` -> % de **pos/neu/neg** (0–1).
* `_sent_by_hour_mean.csv` -> **promedio** de `sent_compound` por hora.
* `_topic_share.csv` -> **proporción** de cada `topic_id` (incluye `-1` si existe).
* `_topic_by_hour.csv` -> **matriz** `topic_id × hour` (solo `topic_id >= 0`).
* `_topic_terms.json` -> **términos top** por tópico (para interpretación).

**Nota:** Si `filterTrafficOnly=True`, el LDA se entrena solo con tweets de tráfico/lluvia; los demás quedan con `topic_id = -1`.

#### Salida en memoria (retorno de `applySection8`)

Diccionario `summary` con:

* `salida_csv`, `tweets`,
* `sent_overall_%`, `sent_by_hour_mean`, `sent_by_rain`, `topic_share_%`,
* `ejemplo_topic_terms` (muestra de términos por tópico).

#### Parámetros `applySection8(...)`

* `inputCsv` *(str)*: ruta del CSV de entrada.
* `outputCsv` *(str, def: `data/clean_traficogt_v4_sent_topics.csv`)*: CSV enriquecido de salida.
* `kTopics` *(int, def: 6)*: número de tópicos LDA.
* `minDf` *(int/float, def: 8)*: **mínimo** de documentos donde debe aparecer un término para incluirse.
* `maxDf` *(int/float, def: 0.6)*: **máximo** de documentos; descarta términos demasiado comunes.
* `ngramRange` *(tuple, def: (1,2))*: usa uni/bigramas (captura frases como `calzada_san_juan`, `zona_10`).
* `extraStop` *(list|set, opcional)*: **stopwords adicionales** específicas del corpus.
* `joinZonaTokens` *(bool, def: True)*: une "zona N / zN" a `zona_N` antes del modelado (mejora señal vial).
* `filterTrafficOnly` *(bool, def: False)*:
  * **False** -> LDA **global** (todo el corpus).
  * **True** -> LDA **focalizado** en tweets de tráfico/lluvia; el resto queda con `topic_id = -1`.
* `saveAux` *(bool, def: True)*: guarda reportes auxiliares (`_sent_overall.csv`, `_topic_share.csv`, etc.).
* `randomState` *(int, def: 42)*: semilla para reproducibilidad (LDA/vectorizador).

In [768]:
def computeAggregates(df: pd.DataFrame) -> Dict[str, dict]:
    """
    Calcula agregados básicos de sentimiento y tópicos.
    Requiere columnas: 'sent_label', 'sent_compound', 'topic_id', 'hour', 'rain_related'
    """
    out = {}

    # Sentimiento global
    sentOverall = df['sent_label'].value_counts(normalize=True).round(3).to_dict()
    out['sent_overall_%'] = sentOverall

    # Sentimiento por hora
    if 'hour' in df.columns:
        sentByHour = df.groupby('hour')['sent_compound'].mean().round(3).to_dict()
        out['sent_by_hour_mean'] = sentByHour

    # Sentimiento vs lluvia
    if 'rain_related' in df.columns:
        tmp = df.groupby('rain_related')['sent_compound'].agg(['mean','count']).round(3)
        out['sent_by_rain'] = tmp.to_dict(orient='index')

    # Share de tópicos
    if 'topic_id' in df.columns:
        topicShare = df['topic_id'].value_counts(normalize=True).sort_index().round(3).to_dict()
        out['topic_share_%'] = topicShare

    return out

In [ ]:
def applySection8(inputCsv: str,
                  outputCsv: str = "data/clean_traficogt_v4.csv",
                  kTopics: int = 6,
                  minDf: int = 8,
                  maxDf: float = 0.6,
                  ngramRange: Tuple[int, int] = (1, 2),
                  extraStop: Optional[List[str]] = None,
                  joinZonaTokens: bool = True,
                  filterTrafficOnly: bool = False,
                  saveAux: bool = True,
                  randomState: int = 42) -> Dict:
    """
    Ejecuta Sección 8: sentimiento + tópicos.
    - inputCsv: CSV con al menos 'date', 'raw_content' y preferible 'lemmas'
    - outputCsv: CSV enriquecido de salida
    - filterTrafficOnly: si True, entrena LDA en el subconjunto de tráfico/lluvia
    - extraStop: lista extra de stopwords del corpus (ej. ["q","vo","ano"])
    - saveAux: si True, guarda CSVs auxiliares de agregados
    """
    df = pd.read_csv(inputCsv)

    # 1) Fecha -> GT y hora
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce').dt.tz_convert('America/Guatemala')
    df['hour'] = df['date'].dt.hour

    # 2) Enlazar zona en texto crudo (no altera tus lemas previos)
    if joinZonaTokens:
        df['raw_content'] = df['raw_content'].fillna('').apply(bindZona)

    # 3) Lemmas
    extraStopSet = set(extraStop) if extraStop else set()
    # Si ya tienes 'lemmas', parsea y limpia:
    if 'lemmas' in df.columns:
        df['lemmas_list'] = df['lemmas'].apply(parseLemmas).apply(lambda xs: cleanLemmaList(xs, extraStop=extraStopSet))
    else:
        # fallback simple si no existen lemas
        df['lemmas_list'] = df['raw_content'].fillna('').str.lower().str.findall(r"[a-záéíóúñ_]+").apply(
            lambda xs: cleanLemmaList(xs, extraStop=extraStopSet)
        )

    # 4) Sentimiento (VADER)
    sia = buildSentimentAnalyzer()
    sentPairs = df['raw_content'].fillna('').apply(lambda t: scoreSentiment(t, sia))
    df['sent_compound'] = sentPairs.apply(lambda x: x[0])
    df['sent_label']    = sentPairs.apply(lambda x: x[1])

    # 5) Flag lluvia
    df['rain_related'] = df['raw_content'].fillna('').apply(isRainRelated)

    # 6) Corpus para tópicos
    df['lemmas_txt'] = df['lemmas_list'].apply(lambda xs: " ".join(xs))
    docsForLda = df['lemmas_txt']

    # 7) Filtrar solo tráfico para el LDA
    if filterTrafficOnly:
        mask = buildTrafficMask(docsForLda)
        docsForLdaUsed = docsForLda[mask]
        idxMap = mask
    else:
        docsForLdaUsed = docsForLda
        idxMap = pd.Series([True]*len(df), index=df.index)

    # 8) Vectorizar y entrenar LDA
    vectorizer, (X, usedMask) = vectorizeForTopics(docsForLdaUsed, minDf=minDf, maxDf=maxDf, ngramRange=ngramRange)
    lda = fitLdaTopics(X, kTopics=kTopics, randomState=randomState)

    # 9) Asignar tópico dominante
    topicIdsUsed = assignDominantTopic(lda, X)
    # Mapear de vuelta a df completo
    df['topic_id'] = np.nan
    df.loc[idxMap[idxMap].index[usedMask], 'topic_id'] = topicIdsUsed
    df['topic_id'] = df['topic_id'].fillna(-1).astype(int)  # -1 = 'sin tópico' (p.ej. doc vacío tras filtros)

    # 10) Resúmenes
    topicTerms = summarizeTopics(lda, vectorizer, topN=10)
    aggregates = computeAggregates(df)

    # 11) Guardar resultados
    df.to_csv(outputCsv, index=False, encoding="utf-8")

    if saveAux:
        # auxiliares típicos para informe
        pd.Series(aggregates.get('sent_overall_%', {})).to_csv(outputCsv.replace(".csv", "_sent_overall.csv"))
        if 'sent_by_hour_mean' in aggregates:
            pd.Series(aggregates['sent_by_hour_mean']).to_csv(outputCsv.replace(".csv", "_sent_by_hour_mean.csv"))
        if 'topic_share_%' in aggregates:
            pd.Series(aggregates['topic_share_%']).to_csv(outputCsv.replace(".csv", "_topic_share.csv"))
        # matriz tópico×hora (solo vistos)
        if (df['topic_id'] >= 0).any():
            topicByHour = df[df['topic_id'] >= 0].groupby(['topic_id','hour']).size().unstack(fill_value=0)
            topicByHour.to_csv(outputCsv.replace(".csv", "_topic_by_hour.csv"))
        # términos por tópico (json)
        with open(outputCsv.replace(".csv", "_topic_terms.json"), "w", encoding="utf-8") as f:
            json.dump(topicTerms, f, ensure_ascii=False, indent=2)

    summary = {
        "salida_csv": outputCsv,
        "tweets": int(len(df)),
        **aggregates,
        "ejemplo_topic_terms": topicTerms
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

## Sección 9

## Pipeline

In [770]:
if DO_S1_2_3:
    processTweetsToCsv(input_path=DATA_FILE, output_path=DATA_CLEAN)
    data_clean1 = applySection2_1()
    data_clean2 = applySection2_2(df=data_clean1)

In [771]:
if DO_S1_2_3:
    exploratoryTrafficAnalysis(df=data_clean2)

In [772]:
if DO_S1_2_3:
    exploratoryTrafficAnalysisVisual(df=data_clean2)

### Volumen y usuarios

* **Tweets totales:** 5,604 de **2,071** usuarios -> **2.71** tweets/usuario en promedio.
  
    *Base:* `numTweets`, `numUsers`.

* **Menciones:** 10,929 -> **1.95** menciones por tweet.
  
    *Base:* `numMentions/numTweets`.

### Interacciones

* **Interacciones totales:** 761,505 -> **135.9** por tweet.
  
    *Base:* suma (replies+RT+quotes+likes)/5,604.

* Composición de interacciones: **Likes 63.5%**, **RT 22.2%**, **Replies 9.7%**, **Quotes 4.6%**.
  
    *Base:* 483,690; 168,891; 74,022; 34,902.

### Hashtags y menciones

* **Top 10 hashtags** suman **27.2%** de todos los hashtags (157/578) -> dispersión alta; el uso de hashtag es bajo (**0.10** por tweet).
  
    *Base:* `numHashtags=578`, top10=157.

* **Nodo central:** `@traficogt` concentra **38.8%** de todas las menciones (4,243/10,929).
  
    *Base:* Top menciones.

* **Concentración:** Top 10 usuarios mencionados = **52.9%** de menciones.
    
    *Base:* 5,784/10,929.

### Horas con más actividad (GT)

* **18:00**: 404 (**7.21%**)
* **13:00**: 389 (**6.94%**)
* **14:00**: 387 (**6.91%**)
* **17:00**: 384 (**6.85%**)
* **19:00**: 363 (**6.48%**)
  
  Estas 5 horas concentran **34.39%** del total (1,927/5,604).
  
  Por franjas: **Tarde 12–17 = 38.26%**, **Noche 18–23 = 30.64%**, **Mañana 06–11 = 27.82%**, **Madrugada 00–05 = 3.28%**.
  
  Pico 18:00 (404) vs mínimo 02:00 (12) => **33.7×** más; vs la **mediana/hora (276)** el pico está **+46.4%**.

### Temas

* Nube de palabras y hashtags mezclan tráfico (*trafico, traficogt*) algo relacionado a política (*guatemala, renunciengolpistas, paronacionalindefinido*).
  
    *Base:* Top hashtags y wordcloud.

### Usuarios influyentes (engagement)

* El ranking de engagement = (likes+RT+replies)/followers muestra **alta concentración**:
    1. **acapelagt** que duplica al segundo, ya que la cola cae rápido ($\approx$ 8–6 en posiciones 3–10).
  
    *Base:* gráfica "Top 10 Usuarios por Engagement".

> **Nota:** Este índice favorece cuentas pequeñas.

### Zonas más mencionadas

* **Zona 1** destaca con **más del doble** de menciones que la segunda (Z10/Z6), seguida por Z9 y Z2; patrón **centrado** en áreas núcleo de la ciudad.
  
    *Base:* regex `\b(Z|Zona)\s*\d{1,2}\b` sobre `raw_content` y gráfica "Top 10 Zonas".

Pasando al análisis zona - hora:

* **Z1 domina**: mayor cobertura horaria y **pico máximo = 6 menciones a las 17:00**; también activa en **05–07 h**.
* **Foco vespertino (15–18 h)**: mayor densidad de celdas !=0 (Z1, Z9, Z10, Z6, Z16).
* **Z9** tiene un **pico 4 a las 17:00** y varios 1–2 en mañana/tarde; **Z10/Z6** aparecen dispersas con valores bajos.
* **Madrugada (0–4 h) y noche tardía (20–23 h)** casi sin menciones.
* Los conteos por celda son **bajos (<=6)** -> patrón indica **horas críticas** más que saturación constante.

In [773]:
if DO_S8:
    summary = applySection8(
        inputCsv=DATA_USE,
        outputCsv=TOPICS_OUT,
        kTopics=10,
        minDf=8,
        maxDf=0.6,
        ngramRange=(1,2),
        extraStop=[],
        joinZonaTokens=True,
        filterTrafficOnly=False, # True: LDA para tweets de tráfico, False: global
        saveAux=True,
        randomState=42
    )

{
  "salida_csv": "data/clean_traficogt_v4.csv",
  "tweets": 5604,
  "sent_overall_%": {
    "neu": 0.63,
    "neg": 0.317,
    "pos": 0.052
  },
  "sent_by_hour_mean": {
    "0": -0.144,
    "1": -0.116,
    "2": -0.095,
    "3": -0.104,
    "4": -0.161,
    "5": -0.169,
    "6": -0.12,
    "7": -0.128,
    "8": -0.12,
    "9": -0.141,
    "10": -0.143,
    "11": -0.135,
    "12": -0.119,
    "13": -0.107,
    "14": -0.122,
    "15": -0.126,
    "16": -0.114,
    "17": -0.136,
    "18": -0.142,
    "19": -0.108,
    "20": -0.111,
    "21": -0.111,
    "22": -0.125,
    "23": -0.096
  },
  "sent_by_rain": {
    "false": {
      "mean": -0.124,
      "count": 5600
    },
    "true": {
      "mean": -0.288,
      "count": 4
    }
  },
  "topic_share_%": {
    "-1": 0.002,
    "0": 0.188,
    "1": 0.095,
    "2": 0.089,
    "3": 0.094,
    "4": 0.091,
    "5": 0.088,
    "6": 0.093,
    "7": 0.073,
    "8": 0.101,
    "9": 0.087
  },
  "ejemplo_topic_terms": {
    "0": [
      "mp",
     

### Temas detectados (LDA **filtrado por tráfico**, `filterTrafficOnly=True`, **K=10**)

*(porcentajes = participación del tópico en todo el corpus; muy baja porque \~94.1% quedó fuera del filtro)*

* **T0 (Villa Lobos / hoyos / accidente)**: *villa, accidente, sur, villa lobo, hoyo, carro* ($\approx$ **0.5%**). **Pico 08:00** (5).
* **T1 (rutas / zona / carril)**: *ruta, zona, pasar, tráfico, amilcarmontejo* ($\approx$ **0.4%**). **Pico 14:00** (5).
* **T2 (PMT / km / incidentes)**: *tráfico, pmt, km, pmtmuniguate, agente* ($\approx$ **0.7%**). **Pico 21:00** (6).
* **T3 (calles / avenidas / zonas)**: *calle, amilcarmontejo, zona, avenida, semáforo* ($\approx$ **0.9%**). **Pico 17:00** (**17**).
* **T4 (avenidas / periférico / carriles)**: *avenida, periférico, zona, carril, transmetro* ($\approx$ **0.4%**). **Pico 15:00** (10).
* **T5 (autoridades / muni / av)**: *av, muniguate, amilcarmontejo, tráfico, calle* ($\approx$ **0.4%**). **Pico 07:00** (10).

> Notas: existen **T6–T9** con cobertura muy baja (**\~0.3–1.2%**). Entre ellos destacan términos de **bloqueos** (*bloqueo, gente, ciudad*) y operativa (*pmt, tránsito, policía*), pero su señal es débil por el tamaño del subconjunto.

### Sentimiento por hora (contexto)

* Promedio **negativo en todas las horas** (subset de tráfico): **05:00 = −0.169** (más negativo) vs **02:00 = −0.095** (menos negativo).

### Lecturas rápidas para @traficogt

* **15–17 h** concentran temas de **calles/avenidas/zonas** y **flujo/carriles** (T3–T4) -> picos operativos.
* **07:00** resalta en **autoridades/muni** (T5) -> inicio de jornada.
* **21:00** sube **incidentes/km/PMT** (T2) -> incidentes nocturnos.

### Temas detectados (LDA **global**, `filterTrafficOnly=False`, **K=10**)

*(porcentajes = participación del tópico en todo el corpus)*

* **T0 (MP / fiscal / corrupción)**: *mp, fiscal, político, consuelo, porra* ($\approx$  **18.8%**). **Pico 17:00** (80).
* **T8 (MPGuatemala / Giammattei / proceso electoral)**: *mpguatemala, drgiammattei, caso, electoral* ($\approx$  **10.1%**). **Pico 14:00** (44).
* **T1 (Arevalo / presidencia / medios)**: *barevalodeleon, presidente, agenciaocote* ($\approx$  **9.5%**). **Pico 13:00** (43).
* **T3 (operativa de tráfico / muni / zonas)**: *amilcarmontejo, trafico, zona, calle* ($\approx$  **9.4%**). **Pico 17:00** (60).
* **T6 (Congreso / diputados / pacto)**: *corrupto, diputado, congreso, ley* ($\approx$  **9.3%**). **Pico 13:00** (46).
* **T4–T5–T7–T9**: conversación **político-social** (prensacomunitar, gobierno, pueblo, democracia…), con picos diurnos (13–19 h) y shares **7–9%** cada uno.

> Lectura central: la agenda **político-judicial domina** la conversación; el bloque vial aparece sobre todo en **T3** (operativa/zonas).

### Sentimiento por hora (contexto)

* Patrón **negativo todo el día**: más negativo **05:00 = −0.169**; menos negativo **02:00 = −0.095**.
